In [ ]:
import json
splits = []
for split_fn in [*snakemake.input.archs4_request_splits, *snakemake.input.cellxgene_request_splits]:  # type: ignore [reportUndefinedVariable]
    with open(split_fn) as f:
        splits.append(json.load(f))

In [ ]:
reference_information = {
    key: value["content"]
    for split in splits
    for key, value in split.items()
}

In [ ]:
# start coding here
# import pandas as pd
import json
import shortuuid

with open(snakemake.input.evaluation_dataset) as f:
    data = json.load(f)

In [ ]:
with open(snakemake.output.formatted_questions, 'w') as qf, open(snakemake.output.reference_responses, "w") as rf:
    for i, d in enumerate(data):   
        question_id = f"{i+1}_{d['id']}"
        json.dump({
            "question_id": question_id,
            "reference": reference_information[d["image"]],
            "text": d["conversations"][0]["value"].replace("<image>", "").strip("\n"),  # stripping to adhere to llava's codebase
            "image": d["image"],
        }, qf)
        qf.write("\n")

        json.dump({"question_id": question_id,
                               "text": d["conversations"][1]["value"],
                               "answer_id": shortuuid.uuid(),
                               "model_id": "gpt-4_with_input_text_and_curation",
                               "metadata": {}}, rf)
        rf.write("\n")